In [64]:
import os
import pandas as pd
import time

# 라이브러리 설치: uv add pydrive2
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# 인증 과정 (만료될 때마다 브라우저 열림)
gauth = GoogleAuth()
gauth.LocalWebserverAuth() 
drive = GoogleDrive(gauth)

def get_subfolder_list(parent_id):
    # 구글 드라이브에게 날릴 질문(Query)을 작성
    # '부모가 parent_id이고, 타입이 폴더이며, 휴지통에 있지 않은 것'
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = False"
    
    file_list = drive.ListFile({'q': query}).GetList()
    
    sub_folders = []
    for f in file_list:
        sub_folders.append({
            'name': f['title'],
            'id': f['id']
        })
    return sub_folders

# 각 폴더 ID를 넣으면 하위 폴더들이 list_of_folders 리스트로 들어감
list_of_folders_tongsin = get_subfolder_list('1LOTaqxasQWNucSTTPn_yGYU7A2D5cDsr')
list_of_folders_card = get_subfolder_list('14u8q6c3z2ndtWlVpI3pOYiIxRLGWHNkS')
list_of_folders_sinyong = get_subfolder_list('1rSiC1tq3dq5MGllTGlzCUhZYB_cnzVrD')
list_of_folders_company = get_subfolder_list('1FQ_6TZ2BxXoAx4Lpg9VNMzj_vlbJCHqV')

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=641874606667-v76q7nr0oa6dgm71k4q9vntrapbaeroq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.


## 기업, 신용
- utf-8-sig
- astype(str) 제거
- on_bad_lines='skip' 제거

In [2]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False
        
        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # df = pd.read_csv(csv_path, encoding=enc, on_bad_lines='skip')
            df = pd.read_csv(csv_path, encoding=enc)
            # df.astype(str).to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except:
            success = False
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except:
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)


### 기업

In [3]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/기업"

for folder_info in list_of_folders_company:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 5. 기업 이전 통계(nps_move_cnt) 작업 시작...

🚀 4. 신규 기업(new) 작업 시작...

🚀 3. 법인 기업(cnt) 작업 시작...


### 신용

In [4]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/신용"

for folder_info in list_of_folders_sinyong:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 9. 이동통계(대민)(WORK_STAT) 작업 시작...

🚀 8. 신용정보(대민)(DM_STAT) 작업 시작...

🚀 4. 전출(대민)(OUT STAT) 작업 시작...

🚀 3. 전입(대민)(IN STAT) 작업 시작...

🚀 7. 전이(경계)(CHAGE_L) 작업 시작...

🚀 6. 전이(CHANGE) 작업 시작...


## 통신 : 
- utf-8-sig 
- astype(str) 제거
- on_bad_lines='skip' 제거
- dtype 지정 (dtype=columns_types)
- engine='pyarrow' 추가
- 에러유발 가능성 큰 컬럼들 사후처리 추가

In [5]:
columns_types = {
    # 식별자 및 코드 (문자열 또는 카테고리)
    'ETL_YM': 'str',  # 기준 년월 : 숫자로 읽으면 앞자리가 사라질 위험이 있으므로 문자열 타입이 가장 안전
    'ETL_YMD': 'str',
    'DOW': 'category', # 요일 구분 --> category
    'SEX_CD': 'category',
    'AGE_GRP': 'category',
    'PURPOSE': 'category',
    'TRANS_GB': 'category',
    
    # 명칭 (문자열)
    'O_MEGA_NM': 'str', 'D_MEGA_NM': 'str',
    'O_CTY_NM': 'str', 'D_CTY_NM': 'str',
    'O_ADMI_NM': 'str', 'D_ADMI_NM': 'str',
    
    # 코드성 숫자 (문자열 권장)
    'O_ADMI_CD': 'str', 'D_ADMI_CD': 'str', # 행정동 코드 : 숫자로 보이지만 실제로는 '코드'이므로 str 타입으로 읽는 것이 표준
    'O_CTY_CD': 'str', 'D_CTY_CD': 'str',
    
    # 시간대 (작은 정수)
    'O_TIME_CD': 'int8',
    'D_TIME_CD': 'int8',
    
    # 수치 데이터 (실수 - 결측치 대응)
    'CNT': 'float32',
    'O_CENTER_X': 'float32',
    'O_CENTER_Y': 'float32',
    'D_CENTER_X': 'float32',
    'D_CENTER_Y': 'float32',
    'DISTANCE': 'float32',
    'CARBON_EMISSIONS': 'float32'
}

def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # 파일 읽기전 0행의 컬럼명만 받아오기
            actual_cols = pd.read_csv(csv_path, nrows=0).columns
            # print(actual_cols) # 0번째 행 컬럼 헤더 깨지지않고 멀쩡한지 확인

            # 실제 파일에 존재하는 컬럼만 골라내어 dtype에 전달
            current_dtypes = {k: v for k, v in columns_types.items() if k in actual_cols}
            df = pd.read_csv(csv_path, encoding=enc, dtype=current_dtypes, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐

            # 에러 유발 가능성이 높은 수치형 컬럼들 사후 처리
            cols_to_fix = ['D_CENTER_Y', 'O_CENTER_Y', 'CNT', 'DISTANCE', 'CARBON_EMISSIONS']
            for col in cols_to_fix:
                if col in df.columns:
                    # 글자가 섞여있어도 NaN으로 바꾸며 float32로 변환 (메모리 절약)
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)

In [6]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/통신"

for folder_info in list_of_folders_tongsin:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 T27 작업 시작...

🚀 T26 작업 시작...

🚀 T25 작업 시작...

🚀 T24 작업 시작...

🚀 T23 작업 시작...

🚀 T22 작업 시작...

🚀 T21 작업 시작...

🚀 T20 작업 시작...

🚀 T18 작업 시작...
📥 다운로드 및 변환 중: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv
❌ seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv 변환 중 에러 발생: CSV parse error: Expected 12 columns, got 13: 202401,월,13,41135,경기도,성남시 분당구,965120,1931210,1,W,05,6764.48,190.61
❌ 변환 실패: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv

🚀 T16 작업 시작...

🚀 T14 작업 시작...

🚀 T13 작업 시작...

🚀 T12 작업 시작...

🚀 T11 작업 시작...

🚀 T10 작업 시작...

🚀 T9 작업 시작...

🚀 T8 작업 시작...

🚀 T7 작업 시작...

🚀 T6 작업 시작...

🚀 T5 작업 시작...

🚀 T4 작업 시작...


In [76]:
# 비정상 데이터 (1건)

# 🚀 T18 작업 시작...
# 📥 다운로드 및 변환 중: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv
# ❌ seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv 변환 중 에러 발생: CSV parse error: Expected 12 columns, got 13: 202401,월,13,41135,경기도,성남시 분당구,965120,1931210,1,W,05,6764.48,190.61
# ❌ 변환 실패: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv

In [ ]:
# 정상
tongsin_2402 = pd.read_parquet('./seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202402_성남시.parquet')

print(f"헤더 컬럼 개수: {len(tongsin_2402.columns)}")
print(f"헤더 목록: {tongsin_2402.columns.tolist()}")

헤더 컬럼 개수: 12
헤더 목록: ['ETL_YM', 'DOW', 'D_CTY_CD', 'D_MEGA_NM', 'D_CTY_NM', 'D_CENTER_X', 'D_CENTER_Y', 'TRANS_GB', 'SEX_CD', 'AGE_GRP', 'CNT', 'DURATION']


In [42]:
# 비정상
# 통신 폴더 내에서 csv->parquet 변환이 안되는 파일 확인
tongsin_2401 = pd.read_csv('seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv')

print(f"헤더 컬럼 개수: {len(tongsin_2401.columns)}")
print(f"헤더 목록: {tongsin_2401.columns.tolist()}")

헤더 컬럼 개수: 12
헤더 목록: ['ETL_YM', 'DOW', 'D_CTY_CD', 'D_MEGA_NM', 'D_CTY_NM', 'D_CENTER_X', 'D_CENTER_Y', 'TRANS_GB', 'SEX_CD', 'AGE_GRP', 'CNT', 'DURATION']


In [39]:
tongsin_2402 # 정상

,ETL_YM,DOW,D_CTY_CD,D_MEGA_NM,D_CTY_NM,D_CENTER_X,D_CENTER_Y,TRANS_GB,SEX_CD,AGE_GRP,CNT,DURATION
0,202402,목,41135,경기도,성남시 분당구,965120.0,1931210.0,3,M,7,44821.621094,221.94
1,202402,토,41135,경기도,성남시 분당구,965120.0,1931210.0,5,W,9,33.189999,138.19
2,202402,일,41131,경기도,성남시 수정구,964981.0,1937423.0,2,M,6,6853.729980,236.12
3,202402,일,41135,경기도,성남시 분당구,965120.0,1931210.0,4,W,6,47.130001,335.82
4,202402,월,41131,경기도,성남시 수정구,964981.0,1937423.0,7,M,6,5554.279785,229.15
...,...,...,...,...,...,...,...,...,...,...,...,...
3079,202402,수,41131,경기도,성남시 수정구,964981.0,1937423.0,2,W,3,20909.939453,263.99
3080,202402,토,41131,경기도,성남시 수정구,964981.0,1937423.0,7,M,5,5163.169922,205.23
3081,202402,화,41131,경기도,성남시 수정구,964981.0,1937423.0,3,M,8,9045.830078,298.03
3082,202402,목,41133,경기도,성남시 중원구,970268.0,1937199.0,7,W,6,6287.459961,264.21


In [43]:
tongsin_2401 # 비정상

,ETL_YM,DOW,D_CTY_CD,D_MEGA_NM,D_CTY_NM,D_CENTER_X,D_CENTER_Y,TRANS_GB,SEX_CD,AGE_GRP,CNT,DURATION
202401,월,13,41135,경기도,성남시 분당구,965120,1931210,1,W,5,6764.48,190.61
202401,화,19,41135,경기도,성남시 분당구,965120,1931210,3,W,6,5479.82,211.64
202401,금,10,41135,경기도,성남시 분당구,965120,1931210,3,M,7,1251.39,244.45
202401,토,7,41133,경기도,성남시 중원구,970268,1937199,1,M,2,88.81,400.31
202401,화,1,41135,경기도,성남시 분당구,965120,1931210,7,M,4,55.00,610.61
...,...,...,...,...,...,...,...,...,...,...,...,...
202401,목,17,41135,경기도,성남시 분당구,965120,1931210,5,W,4,23.40,248.90
202401,월,2,41131,경기도,성남시 수정구,964981,1937423,2,W,6,8.30,423.88
202401,일,9,41131,경기도,성남시 수정구,964981,1937423,7,W,5,155.48,309.47
202401,월,0,41131,경기도,성남시 수정구,964981,1937423,7,W,8,3.43,255.09


In [ ]:
# 비정상 테이블의 비정상 컬럼이 T16의 D_TIME과 같은 의미를 가지고 있는지 확인
tongsin_16_2401 = pd.read_parquet('./seongnam_data/통신/T16/T16_GG_PURPOSE_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.parquet')
tongsin_16_2401

,ETL_YM,DOW,D_TIME,D_CTY_CD,D_MEGA_NM,D_CTY_NM,D_CENTER_X,D_CENTER_Y,PURPOSE,SEX_CD,AGE_GRP,CNT,DURATION
0,202401,수,21,41131,경기도,성남시 수정구,964981.0,1937423.0,1,W,2,3.110000,54.18
1,202401,토,6,41135,경기도,성남시 분당구,965120.0,1931210.0,1,M,6,810.750000,429.00
2,202401,목,13,41133,경기도,성남시 중원구,970268.0,1937199.0,0,M,5,747.750000,398.79
3,202401,화,17,41131,경기도,성남시 수정구,964981.0,1937423.0,0,M,1,288.000000,311.02
4,202401,월,20,41135,경기도,성남시 분당구,965120.0,1931210.0,2,W,4,61.209999,168.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33419,202401,화,13,41135,경기도,성남시 분당구,965120.0,1931210.0,6,M,4,11377.250000,161.07
33420,202401,월,13,41135,경기도,성남시 분당구,965120.0,1931210.0,0,M,7,2819.270020,396.35
33421,202401,목,17,41131,경기도,성남시 수정구,964981.0,1937423.0,1,M,2,15.600000,304.44
33422,202401,토,9,41131,경기도,성남시 수정구,964981.0,1937423.0,0,M,7,371.649994,453.45


In [ ]:
# t16의 d_time 컬럼과 t18 오류데이터의 dow 컬럼이 같은 의미를 담고 있는 컬럼인지 확인

# t16 
print(tongsin_16_2401['D_TIME'].value_counts().sort_index())

# t18 비정상
print(tongsin_2401['DOW'].value_counts().sort_index())

# 전체 행수 비교 : 다르기 때문에 비교 불가 (테이블 목적 자체가 다르기 때문에 의미 없음)
print(len(tongsin_16_2401))
print(len(tongsin_2401))

D_TIME
0      847
1     1103
2     1094
3     1123
4     1171
5     1261
6     1341
7     1401
8     1515
9     1547
10    1553
11    1574
12    1565
13    1565
14    1571
15    1555
16    1561
17    1532
18    1502
19    1479
20    1446
21    1441
22    1423
23    1254
Name: count, dtype: int64
DOW
0     1238
1     1618
2     1586
3     1552
4     1614
5     1737
6     1899
7     2025
8     2154
9     2253
10    2336
11    2427
12    2485
13    2489
14    2421
15    2495
16    2569
17    2582
18    2551
19    2516
20    2522
21    2472
22    2462
23    2229
Name: count, dtype: int64
33424
52232


- t18과 t16의 컬럼 의미가 같다하더라도 두 테이블의 목적 자체가 달라서 어차피 의미 없음
- t18의 나머지 테이블에선 '시간대'에 관한 컬럼은 존재하지 않으므로 컬럼 제거하는 것이 적합하다고 판단.

In [ ]:
tongsin_2402.columns # 정상 테이블 컬럼

Index(['ETL_YM', 'DOW', 'D_CTY_CD', 'D_MEGA_NM', 'D_CTY_NM', 'D_CENTER_X',
       'D_CENTER_Y', 'TRANS_GB', 'SEX_CD', 'AGE_GRP', 'CNT', 'DURATION'],
      dtype='str')

In [34]:
tongsin_2401 = pd.read_csv('seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv')
tongsin_2401.columns.tolist()

['ETL_YM',
 'DOW',
 'D_CTY_CD',
 'D_MEGA_NM',
 'D_CTY_NM',
 'D_CENTER_X',
 'D_CENTER_Y',
 'TRANS_GB',
 'SEX_CD',
 'AGE_GRP',
 'CNT',
 'DURATION']

In [45]:
# 통신 T18 > 202401 비정상 데이터 처리

correct_columns = [ # 정상 컬럼명 지정
    "ETL_YM", "DOW", "D_TIME", "D_CTY_CD", "D_MEGA_NM", "D_CTY_NM",
    "D_CENTER_X", "D_CENTER_Y", "TRANS_GB", "SEX_CD",
    "AGE_GRP", "CNT", "DURATION"
]
tongsin_2401 = pd.read_csv('seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv', 
                            header=0,               # 기존 헤더 행을 덮어씀
                            names=correct_columns)  # 13개 컬럼명 직접 지정 

print(f"컬럼 수: {len(tongsin_2401.columns)}")
print(tongsin_2401.head())

컬럼 수: 13
   ETL_YM DOW  D_TIME  D_CTY_CD D_MEGA_NM D_CTY_NM  D_CENTER_X  D_CENTER_Y  \
0  202401   월      13     41135       경기도  성남시 분당구      965120     1931210   
1  202401   화      19     41135       경기도  성남시 분당구      965120     1931210   
2  202401   금      10     41135       경기도  성남시 분당구      965120     1931210   
3  202401   토       7     41133       경기도  성남시 중원구      970268     1937199   
4  202401   화       1     41135       경기도  성남시 분당구      965120     1931210   

   TRANS_GB SEX_CD  AGE_GRP      CNT  DURATION  
0         1      W        5  6764.48    190.61  
1         3      W        6  5479.82    211.64  
2         3      M        7  1251.39    244.45  
3         1      M        2    88.81    400.31  
4         7      M        4    55.00    610.61  


In [46]:
tongsin_2401 = tongsin_2401.drop(columns='D_TIME')
print(f"최종 컬럼 수: {len(tongsin_2401.columns)}")  # 12개 확인
print(tongsin_2401.head())

최종 컬럼 수: 12
   ETL_YM DOW  D_CTY_CD D_MEGA_NM D_CTY_NM  D_CENTER_X  D_CENTER_Y  TRANS_GB  \
0  202401   월     41135       경기도  성남시 분당구      965120     1931210         1   
1  202401   화     41135       경기도  성남시 분당구      965120     1931210         3   
2  202401   금     41135       경기도  성남시 분당구      965120     1931210         3   
3  202401   토     41133       경기도  성남시 중원구      970268     1937199         1   
4  202401   화     41135       경기도  성남시 분당구      965120     1931210         7   

  SEX_CD  AGE_GRP      CNT  DURATION  
0      W        5  6764.48    190.61  
1      W        6  5479.82    211.64  
2      M        7  1251.39    244.45  
3      M        2    88.81    400.31  
4      M        4    55.00    610.61  


In [47]:
# 비정상 데이터 parquet로 변환
csv_path = 'seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv'
parquet_path = csv_path.replace('.csv', '.parquet')
tongsin_2401.to_parquet(parquet_path, engine='pyarrow', compression='snappy')

In [50]:
tongsin_2401_csv = pd.read_csv('seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv')
tongsin_2401_par = pd.read_parquet('seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.parquet')

tongsin_2401_csv.head()

,ETL_YM,DOW,D_CTY_CD,D_MEGA_NM,D_CTY_NM,D_CENTER_X,D_CENTER_Y,TRANS_GB,SEX_CD,AGE_GRP,CNT,DURATION
202401,월,13,41135,경기도,성남시 분당구,965120,1931210,1,W,5,6764.48,190.61
202401,화,19,41135,경기도,성남시 분당구,965120,1931210,3,W,6,5479.82,211.64
202401,금,10,41135,경기도,성남시 분당구,965120,1931210,3,M,7,1251.39,244.45
202401,토,7,41133,경기도,성남시 중원구,970268,1937199,1,M,2,88.81,400.31
202401,화,1,41135,경기도,성남시 분당구,965120,1931210,7,M,4,55.00,610.61


In [ ]:
tongsin_2401_par.head() # 데이터 변환 성공

,ETL_YM,DOW,D_CTY_CD,D_MEGA_NM,D_CTY_NM,D_CENTER_X,D_CENTER_Y,TRANS_GB,SEX_CD,AGE_GRP,CNT,DURATION
0,202401,월,41135,경기도,성남시 분당구,965120,1931210,1,W,5,6764.48,190.61
1,202401,화,41135,경기도,성남시 분당구,965120,1931210,3,W,6,5479.82,211.64
2,202401,금,41135,경기도,성남시 분당구,965120,1931210,3,M,7,1251.39,244.45
3,202401,토,41133,경기도,성남시 중원구,970268,1937199,1,M,2,88.81,400.31
4,202401,화,41135,경기도,성남시 분당구,965120,1931210,7,M,4,55.00,610.61


## 카드
- cp949
- astype(str) 제거
- on_bad_lines='skip' 제거
- engine='pyarrow' 추가
- sep='|' 추가

In [67]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 cp949로 지정
        enc = 'cp949'
        try:
            df = pd.read_csv(csv_path, sep='|', encoding=enc, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)

In [68]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/카드"

for folder_info in list_of_folders_card:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 10. 매출(대민)(day) 작업 시작...
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2404_성남시.csv
❌ seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2404_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 129: illegal multibyte sequence
❌ 변환 실패: tbsh_gyeonggi_day_2404_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2403_성남시.csv
❌ seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2403_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 129: illegal multibyte sequence
❌ 변환 실패: tbsh_gyeonggi_day_2403_성남시.csv

🚀 3. 가맹점 정보(대민)(mer_s) 작업 시작...
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2404_성남시.csv
❌ seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2404_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 256: illegal multibyte sequence
❌ 변환 실패: tbsh_gyeonggi_mer_s_2404_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2403_성남시.csv
❌ seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2403_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 256: illega

In [ ]:
# 비정상 데이터 (4건)

# 🚀 10. 매출(대민)(day) 작업 시작...
# 📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2404_성남시.csv
# ❌ seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2404_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 129: illegal multibyte sequence
# ❌ 변환 실패: tbsh_gyeonggi_day_2404_성남시.csv
# 📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2403_성남시.csv
# ❌ seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2403_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 129: illegal multibyte sequence
# ❌ 변환 실패: tbsh_gyeonggi_day_2403_성남시.csv

# 🚀 3. 가맹점 정보(대민)(mer_s) 작업 시작...
# 📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2404_성남시.csv
# ❌ seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2404_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 256: illegal multibyte sequence
# ❌ 변환 실패: tbsh_gyeonggi_mer_s_2404_성남시.csv
# 📥 다운로드 및 변환 중: tbsh_gyeonggi_mer_s_2403_성남시.csv
# ❌ seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2403_성남시.csv 변환 중 에러 발생: 'cp949' codec can't decode byte 0xec in position 256: illegal multibyte sequence
# ❌ 변환 실패: tbsh_gyeonggi_mer_s_2403_성남시.csv

In [ ]:
day_2403 = pd.read_csv('./seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2403_성남시.csv') # 비정상
day_2403

,ta_ymd,cty_rgn_no,admi_cty_no,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,hour,sex,age,day,amt,cnt
0,20240301,41131,41131510,D01,소매/유통,가전제품,4,F,8,5,4068503,2
1,20240301,41131,41131510,D01,소매/유통,가전제품,5,F,4,5,3012048,2
2,20240301,41131,41131510,D01,소매/유통,가전제품,5,F,5,5,25288,2
3,20240301,41131,41131510,D01,소매/유통,가전제품,5,F,7,5,537866,2
4,20240301,41131,41131510,D01,소매/유통,가전제품,5,M,5,5,43029,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2497575,20240331,41135,41135680,Y04,공공/기업/단체,단체,5,F,7,7,118227,3
2497576,20240331,41135,41135680,Y04,공공/기업/단체,단체,6,F,6,7,77759,3
2497577,20240331,41135,41135680,Y04,공공/기업/단체,단체,7,F,4,7,32191,2
2497578,20240331,41135,41135680,Y04,공공/기업/단체,단체,7,F,7,7,26087,2


In [70]:
day_2404 = pd.read_csv('./seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2404_성남시.csv')
day_2404

,ta_ymd,cty_rgn_no,admi_cty_no,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,hour,sex,age,day,amt,cnt
0,20240401,41131,41131510,D01,소매/유통,가전제품,4,F,4,1,10757,2
1,20240401,41131,41131510,D01,소매/유통,가전제품,5,F,6,1,358692,2
2,20240401,41131,41131510,D01,소매/유통,가전제품,5,F,7,1,753012,2
3,20240401,41131,41131510,D01,소매/유통,가전제품,5,F,8,1,715663,4
4,20240401,41131,41131510,D01,소매/유통,가전제품,5,M,5,1,107358,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2460080,20240430,41135,41135680,Y04,공공/기업/단체,단체,7,M,4,2,8361,2
2460081,20240430,41135,41135680,Y04,공공/기업/단체,단체,7,M,5,2,8361,2
2460082,20240430,41135,41135680,Y04,공공/기업/단체,단체,7,M,6,2,24833,2
2460083,20240430,41135,41135680,Y04,공공/기업/단체,단체,7,M,7,2,8361,2


In [ ]:
mers_2403 = pd.read_csv('./seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2403_성남시.csv') # 비정상
mers_2403

,ta_ym,cty_rgn_no,admi_cty_no,blk_cd,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,sale,mm_cnt,mer_cnt,fran_cnt,open_cnt,stop_cnt,close_cnt,open_11_cnt,open_12_23_cnt,open_24_35_cnt,open_36_47_cnt,open_48_59_cnt,open_60_cnt
0,202403,41135,41135510,100470,Q13,음식,커피/음료,A,71,1,0,0,1,0,0,0,0,0,0,1
1,202403,41135,41135510,100470,R07,학문/교육,입시학원,A,98,1,1,0,1,0,0,0,0,0,0,1
2,202403,41135,41135680,101678,O03,여가/오락,일반스포츠,A,46,2,0,0,1,0,0,1,1,0,0,0
3,202403,41135,41135662,101808,D08,소매/유통,음/식료품소매,신규,0,1,0,1,1,0,0,0,0,0,0,0
4,202403,41135,41135662,101808,R06,학문/교육,유아교육,A,142,1,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56078,202403,41135,41135670,99971,D11,소매/유통,종합소매점,NaN,116,1,1,0,0,0,0,0,0,0,0,1
56079,202403,41135,41135670,99971,D19,소매/유통,제조/도매,A,213,2,0,0,2,0,0,0,0,0,0,0
56080,202403,41135,41135670,99971,F06,생활서비스,세탁/가사서비스,A,179,1,0,0,0,0,0,0,0,0,0,1
56081,202403,41135,41135670,99971,F17,생활서비스,교통서비스,NaN,213,1,0,0,1,0,0,0,0,0,0,0


In [73]:
mers_2404 = pd.read_csv('./seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2404_성남시.csv')
mers_2404

,ta_ym,cty_rgn_no,admi_cty_no,blk_cd,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,sale,mm_cnt,mer_cnt,fran_cnt,open_cnt,stop_cnt,close_cnt,open_11_cnt,open_12_23_cnt,open_24_35_cnt,open_36_47_cnt,open_48_59_cnt,open_60_cnt
0,202404,41135,41135510,100470,Q13,음식,커피/음료,A,72,1,0,0,1,0,0,0,0,0,0,1
1,202404,41135,41135510,100470,R07,학문/교육,입시학원,A,99,1,1,0,1,0,0,0,0,0,0,1
2,202404,41135,41135680,101678,O03,여가/오락,일반스포츠,A,46,2,0,0,1,0,0,1,1,0,0,0
3,202404,41135,41135662,101808,D08,소매/유통,음/식료품소매,신규,0,1,0,1,1,0,0,0,0,0,0,0
4,202404,41135,41135662,101808,R06,학문/교육,유아교육,A,142,1,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56193,202404,41135,41135670,99971,D19,소매/유통,제조/도매,A,215,2,0,0,2,0,0,0,0,0,0,0
56194,202404,41135,41135670,99971,F06,생활서비스,세탁/가사서비스,A,180,1,0,0,1,0,0,0,0,0,0,0
56195,202404,41135,41135670,99971,F17,생활서비스,교통서비스,A,214,1,0,0,1,0,0,0,0,0,0,0
56196,202404,41135,41135670,99971,Q03,음식,닭/오리요리,A,146,1,0,0,0,0,0,0,0,0,0,1


In [75]:
# 비정상 데이터 
csv_path = [
    './seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2403_성남시.csv',
    './seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2404_성남시.csv',
    './seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2403_성남시.csv',
    './seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2404_성남시.csv'
]
for path in csv_path:
    df = pd.read_csv(path, encoding='utf-8-sig') # sep='|' 옵션 제거, 인코딩 방식 utf-8-sig로 지정
    parquet_path = path.replace('.csv', '.parquet')
    df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
    os.remove(path) # 성공 즉시 CSV 삭제 (용량 확보)